# Basic imports

In [1]:
# importing libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings 
warnings.filterwarnings("ignore")


#importing wordcloud for text visualization
from wordcloud import WordCloud

#import nltk for natural language processing
import nltk 
from nltk.corpus import stopwords

#downnloading nltk data
nltk.download('stopwords')
nltk.download('punkt')

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\samir\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\samir\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!


True

In [2]:
df=pd.read_csv('spam.csv')
df.head()

,v1,v2,Unnamed: 2,Unnamed: 3,Unnamed: 4
0,ham,"Go until jurong point, crazy.. Available only ...",NaN,NaN,NaN
1,ham,Ok lar... Joking wif u oni...,NaN,NaN,NaN
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...,NaN,NaN,NaN
3,ham,U dun say so early hor... U c already then say...,NaN,NaN,NaN
4,ham,"Nah I don't think he goes to usf, he lives aro...",NaN,NaN,NaN


In [3]:
df.drop(columns=['Unnamed: 2','Unnamed: 3','Unnamed: 4'],inplace=True)
df.head()

,v1,v2
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."


In [4]:
# renaming columns for easiness
df.rename(columns={'v1':'target','v2':'text'},inplace=True)
df.head()


,target,text
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."


# data preprocessing

In [5]:
# encoding target columns into numbers
from sklearn.preprocessing import LabelEncoder
encoder=LabelEncoder()
df['target'] = encoder.fit_transform(df['target'])
df.head()

,target,text
0,0,"Go until jurong point, crazy.. Available only ..."
1,0,Ok lar... Joking wif u oni...
2,1,Free entry in 2 a wkly comp to win FA Cup fina...
3,0,U dun say so early hor... U c already then say...
4,0,"Nah I don't think he goes to usf, he lives aro..."


In [6]:
# cheking data imbalance
df.target.value_counts()/len(df) *100

target
0    86.593683
1    13.406317
Name: count, dtype: float64

In [7]:
#checking duplicated data
df.duplicated().sum()

np.int64(403)

In [8]:
#remove duplicated data
df=df.drop_duplicates(keep='first')
len(df)

5169

In [9]:
df.info()

<class 'pandas.DataFrame'>
Index: 5169 entries, 0 to 5571
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype
---  ------  --------------  -----
 0   target  5169 non-null   int64
 1   text    5169 non-null   str  
dtypes: int64(1), str(1)
memory usage: 121.1 KB


# Feature Engineering

In [10]:
# importing the porter stemmer for text stemming
from nltk.stem.porter import PorterStemmer

#importing the string module for handling special character 
import string

# creating an instance of the porterstemmer

ports=PorterStemmer()

In [11]:
# transforming the text into lower case text preprocessing
nltk.download('punkt_tab')

def transform_text(text):
    #transform text into lower case
    text=text.lower()

    #tokenization using nltk
    text=nltk.word_tokenize(text)

    # removing special characters
    cleaned_text=[]
    for i in text:
        if i.isalnum(): #is the i madeup of letters or not 
            cleaned_text.append(i)

    # removing stopwords and punctuations
    text=cleaned_text[:] #copying data from cleaned_text
    cleaned_text.clear()

    #loop through the tokens and remove stopwords  and punctuations
    for i in text:
        if i not in stopwords.words('english') and i not in string.punctuation:
            cleaned_text.append(i)

    #stemming using porter stemmer
    text=cleaned_text[:] # agian copying data from cleaned_text
    cleaned_text.clear()
    for i in text:
        cleaned_text.append(ports.stem(i))

    # joinn the processed tokens back to  a single string 
    return " ".join(cleaned_text)


[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\samir\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


In [12]:
transform_text("The students are studying Python programming! They were playing games.")

'student studi python program play game'

In [13]:
df['transformed_text']=df['text'].apply(transform_text)
df.head()

,target,text,transformed_text
0,0,"Go until jurong point, crazy.. Available only ...",go jurong point crazi avail bugi n great world...
1,0,Ok lar... Joking wif u oni...,ok lar joke wif u oni
2,1,Free entry in 2 a wkly comp to win FA Cup fina...,free entri 2 wkli comp win fa cup final tkt 21...
3,0,U dun say so early hor... U c already then say...,u dun say earli hor u c alreadi say
4,0,"Nah I don't think he goes to usf, he lives aro...",nah think goe usf live around though


In [14]:
# feature engineering

from sklearn.feature_extraction.text import CountVectorizer ,TfidfVectorizer
tfid=TfidfVectorizer(max_features=500)


In [15]:
print(df.target.values)
print(df.target.value_counts())

[0 0 1 ... 0 0 0]
target
0    4516
1     653
Name: count, dtype: int64


In [16]:
X=tfid.fit_transform(df['transformed_text']).toarray()
y=df['target'].values

In [17]:
#train test split
from sklearn.model_selection import train_test_split
X_train , X_test ,y_train , y_test =train_test_split(X,y,test_size=0.20,random_state=2)

In [18]:
print(X_train.shape)
print(y_train.shape)

(4135, 500)
(4135,)


# model training


In [27]:
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.naive_bayes import MultinomialNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import AdaBoostClassifier
from sklearn.ensemble import BaggingClassifier
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.ensemble import GradientBoostingClassifier
from xgboost import XGBClassifier


In [33]:
svc=SVC(kernel='sigmoid',gamma=1.0)
knc=KNeighborsClassifier()
mnb=MultinomialNB()
dtc=DecisionTreeClassifier(max_depth=6)
logi=LogisticRegression(solver='liblinear',penalty='l1')
rfc=RandomForestClassifier(n_estimators=50,random_state=2)
abc=AdaBoostClassifier(n_estimators=50,random_state=2)
bc=BaggingClassifier(n_estimators=50,random_state=2)
etc=ExtraTreesClassifier(n_estimators=50,random_state=2)
gbdt=GradientBoostingClassifier(n_estimators=50,random_state=2)
xgb=XGBClassifier(n_estimator=50,random_state=2)

In [34]:
clf={
    'SVC':svc,
    'KNN':knc,
    'NB':mnb,
    'DT':dtc,
    'LR':logi,
    'RF':rfc,
    'Adaboost':abc,
    'Bgc':bc,
    'ETC':etc,
    'GBDT':gbdt,
    'xgb':xgb

}

# model evaluation

In [35]:
from sklearn.metrics import accuracy_score , precision_score
def train_classifier(clfs,X_train,y_train,X_test,y_test):
    clfs.fit(X_train,y_train)

    y_pred=clfs.predict(X_test)

    accuracy=accuracy_score(y_test,y_pred)

    precision=precision_score(y_test,y_pred)
    
    return accuracy , precision

In [36]:
accuracy_scores=[]
precision_scores=[]

for  name,clfs in clf.items():
    current_acc,current_precision=train_classifier(clfs,X_train,y_train,X_test,y_test)

    print()

    print("for:",name)
    print("accuracy is ",current_acc)
    print("preision is ",current_precision)

    accuracy_scores.append(current_acc)
    precision_scores.append(current_precision)


for: SVC
accuracy is  0.9680851063829787
preision is  0.9487179487179487

for: KNN
accuracy is  0.9274661508704062
preision is  1.0

for: NB
accuracy is  0.9709864603481625
preision is  0.9655172413793104

for: DT
accuracy is  0.9439071566731141
preision is  0.9081632653061225

for: LR
accuracy is  0.9632495164410058
preision is  0.9629629629629629

for: RF
accuracy is  0.9729206963249516
preision is  0.9435483870967742

for: Adaboost
accuracy is  0.9235976789168279
preision is  0.8734177215189873

for: Bgc
accuracy is  0.9593810444874274
preision is  0.8870967741935484

for: ETC
accuracy is  0.971953578336557
preision is  0.9291338582677166

for: GBDT
accuracy is  0.9526112185686654
preision is  0.9405940594059405

for: xgb
accuracy is  0.9729206963249516
preision is  0.9583333333333334


In [37]:
print(accuracy_scores)
print(precision_scores)

[0.9680851063829787, 0.9274661508704062, 0.9709864603481625, 0.9439071566731141, 0.9632495164410058, 0.9729206963249516, 0.9235976789168279, 0.9593810444874274, 0.971953578336557, 0.9526112185686654, 0.9729206963249516]
[0.9487179487179487, 1.0, 0.9655172413793104, 0.9081632653061225, 0.9629629629629629, 0.9435483870967742, 0.8734177215189873, 0.8870967741935484, 0.9291338582677166, 0.9405940594059405, 0.9583333333333334]
